# predicting any condition from c4 dataset

# imports and setup

In [ ]:
# Predicting ANY Mental Health or Neurodevelopmental Condition
# Comprehensive ML Model Training and Feature Engineering

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

print("=== ANY CONDITION PREDICTION MODEL TRAINING ===")
print("Dataset: C4 Raw Data")
print("Target: any_condition_target (binary classification)")
print("Groups: 1. Healthy Controls, 2. Any Diagnosis")

# data loading and inital dora

In [ ]:
# Load the raw data
raw_data = pd.read_csv('/Users/eb2007/Library/CloudStorage/OneDrive-UniversityofCambridge/Documents/PhD/data/data_c4_raw.csv')

# Remove test entries (first 16 rows)
data = raw_data.iloc[16:].reset_index(drop=True)

print(f"Dataset shape: {data.shape}")

# Get diagnosis columns
diagnosis_cols = [col for col in data.columns if col.startswith('diagnosis_') and col != 'diagnosis_9']
autism_diagnosis_cols = [col for col in data.columns if col.startswith('autism_diagnosis_')]

print(f"Diagnosis columns: {diagnosis_cols}")
print(f"Autism diagnosis columns: {autism_diagnosis_cols}")

# SIMPLE LOGIC:
# 1. Find people with any diagnosis (values 1-7 in diagnosis columns)
diagnosis_mask = pd.Series([False] * len(data))
for col in diagnosis_cols:
    if col in data.columns:
        diagnosis_mask = diagnosis_mask | data[col].isin([1, 2, 3, 4, 5, 6, 7])

# 2. Find people with autism diagnosis (any value in autism_diagnosis columns)
# BUT only if they don't already have autism (value 2) in diagnosis columns
already_autism_mask = pd.Series([False] * len(data))
for col in diagnosis_cols:
    if col in data.columns:
        already_autism_mask = already_autism_mask | (data[col] == 2)

autism_diagnosis_mask = pd.Series([False] * len(data))
for col in autism_diagnosis_cols:
    if col in data.columns:
        autism_diagnosis_mask = autism_diagnosis_mask | (data[col].notna() & ~already_autism_mask)

# 3. Combine all diagnosis cases
all_diagnosis_mask = diagnosis_mask | autism_diagnosis_mask

# 4. Healthy controls = everyone else (but exclude "prefer not to say")
prefer_not_say_mask = pd.Series([False] * len(data))
for col in diagnosis_cols:
    if col in data.columns:
        prefer_not_say_mask = prefer_not_say_mask | (data[col] == 8)

healthy_mask = ~all_diagnosis_mask & ~prefer_not_say_mask

print(f"People with any diagnosis: {all_diagnosis_mask.sum()}")
print(f"Healthy controls: {healthy_mask.sum()}")
print(f"Prefer not to say: {prefer_not_say_mask.sum()}")

# 5. Create balanced dataset
diagnosis_data = data[all_diagnosis_mask].copy()
healthy_data = data[healthy_mask].copy()

print(f"\nDiagnosis data shape: {diagnosis_data.shape}")
print(f"Healthy data shape: {healthy_data.shape}")

# Sample equal numbers from each group
min_samples = min(len(diagnosis_data), len(healthy_data))
print(f"Minimum samples: {min_samples}")

diagnosis_balanced = diagnosis_data.sample(n=min_samples, random_state=42)
healthy_balanced = healthy_data.sample(n=min_samples, random_state=42)

# Combine and shuffle
balanced_data = pd.concat([healthy_balanced, diagnosis_balanced], ignore_index=True)
balanced_data = balanced_data.sample(frac=1, random_state=42).reset_index(drop=True)

# Create target variable
balanced_data['any_condition_target'] = 0  # Healthy controls
balanced_data.loc[len(healthy_balanced):, 'any_condition_target'] = 1  # Any diagnosis

print(f"\nFinal balanced dataset shape: {balanced_data.shape}")
print(f"Target distribution: {balanced_data['any_condition_target'].value_counts()}")

# Save balanced dataset
balanced_data.to_csv('/Users/eb2007/playground/bullpy/c4_experiments/data/processed/data_c4_any_condition_balanced.csv', index=False)
print("Saved balanced dataset")

# VERIFICATION - Test the logic on a few examples
print(f"\n=== VERIFICATION ===")

# Test the logic on a few rows
test_rows = balanced_data.head(10)
for idx, row in test_rows.iterrows():
    has_diagnosis = False
    diagnoses = []
    
    # Check diagnosis columns
    for col in diagnosis_cols:
        if col in row.index and not pd.isna(row[col]):
            if row[col] in [1, 2, 3, 4, 5, 6, 7]:
                has_diagnosis = True
                diagnoses.append(f"{col}={row[col]}")
    
    # Check autism diagnosis columns
    for col in autism_diagnosis_cols:
        if col in row.index and not pd.isna(row[col]):
            has_diagnosis = True
            diagnoses.append(f"{col}=autism")
    
    expected = row['any_condition_target'] == 1
    status = "✓" if has_diagnosis == expected else "✗"
    print(f"Person {idx}: {status} (has_diagnosis={has_diagnosis}, expected={expected}, diagnoses={diagnoses})")

# understanding diagnosis columns 

In [ ]:
# Identify diagnosis columns (I to Q based on variable key)
diagnosis_cols = []
for col in data.columns:
    if any(term in col.lower() for term in ['adhd', 'autism', 'bipolar', 'depression', 'learning', 'ocd', 'schizophrenia', 'diagnosed']):
        diagnosis_cols.append(col)

print(f"Found diagnosis columns: {diagnosis_cols}")

# Check diagnosis column values
for col in diagnosis_cols:
    if col in data.columns:
        print(f"\n{col} value counts:")
        print(data[col].value_counts().head(10))
        print(f"Missing values: {data[col].isnull().sum()}")

# 4. create balanced dataset

In [ ]:
# Load the raw data
raw_data = pd.read_csv('/Users/eb2007/Library/CloudStorage/OneDrive-UniversityofCambridge/Documents/PhD/data/data_c4_raw.csv')

# Remove test entries (first 16 rows)
data = raw_data.iloc[16:].reset_index(drop=True)

print(f"Dataset shape: {data.shape}")

# Get diagnosis columns
diagnosis_cols = [col for col in data.columns if col.startswith('diagnosis_') and col != 'diagnosis_9']
autism_diagnosis_cols = [col for col in data.columns if col.startswith('autism_diagnosis_')]

print(f"Diagnosis columns: {diagnosis_cols}")
print(f"Autism diagnosis columns: {autism_diagnosis_cols}")

# SIMPLE LOGIC:
# 1. Find people with any diagnosis (values 1-7 in diagnosis columns)
diagnosis_mask = pd.Series([False] * len(data))
for col in diagnosis_cols:
    if col in data.columns:
        diagnosis_mask = diagnosis_mask | data[col].isin([1, 2, 3, 4, 5, 6, 7])

# 2. Find people with autism diagnosis (any value in autism_diagnosis columns)
# BUT only if they don't already have autism (value 2) in diagnosis columns
already_autism_mask = pd.Series([False] * len(data))
for col in diagnosis_cols:
    if col in data.columns:
        already_autism_mask = already_autism_mask | (data[col] == 2)

autism_diagnosis_mask = pd.Series([False] * len(data))
for col in autism_diagnosis_cols:
    if col in data.columns:
        autism_diagnosis_mask = autism_diagnosis_mask | (data[col].notna() & ~already_autism_mask)

# 3. Combine all diagnosis cases
all_diagnosis_mask = diagnosis_mask | autism_diagnosis_mask

# 4. Healthy controls = everyone else (but exclude "prefer not to say")
prefer_not_say_mask = pd.Series([False] * len(data))
for col in diagnosis_cols:
    if col in data.columns:
        prefer_not_say_mask = prefer_not_say_mask | (data[col] == 8)

healthy_mask = ~all_diagnosis_mask & ~prefer_not_say_mask

print(f"People with any diagnosis: {all_diagnosis_mask.sum()}")
print(f"Healthy controls: {healthy_mask.sum()}")
print(f"Prefer not to say: {prefer_not_say_mask.sum()}")

# 5. Create balanced dataset
diagnosis_data = data[all_diagnosis_mask].copy()
healthy_data = data[healthy_mask].copy()

print(f"\nDiagnosis data shape: {diagnosis_data.shape}")
print(f"Healthy data shape: {healthy_data.shape}")

# Sample equal numbers from each group
min_samples = min(len(diagnosis_data), len(healthy_data))
print(f"Minimum samples: {min_samples}")

diagnosis_balanced = diagnosis_data.sample(n=min_samples, random_state=42)
healthy_balanced = healthy_data.sample(n=min_samples, random_state=42)

# Combine WITHOUT shuffling to keep groups separate
balanced_data = pd.concat([healthy_balanced, diagnosis_balanced], ignore_index=True)

# Create target variable BEFORE any shuffling
balanced_data['any_condition_target'] = 0  # Healthy controls (first half)
balanced_data.loc[len(healthy_balanced):, 'any_condition_target'] = 1  # Any diagnosis (second half)

print(f"\nFinal balanced dataset shape: {balanced_data.shape}")
print(f"Target distribution: {balanced_data['any_condition_target'].value_counts()}")

# Save balanced dataset
balanced_data.to_csv('/Users/eb2007/playground/bullpy/c4_experiments/data/processed/data_c4_any_condition_balanced.csv', index=False)
print("Saved balanced dataset")

# VERIFICATION - Test the logic on a few examples
print(f"\n=== VERIFICATION ===")

# Test the logic on a few rows
test_rows = balanced_data.head(10)
for idx, row in test_rows.iterrows():
    has_diagnosis = False
    diagnoses = []
    
    # Check diagnosis columns
    for col in diagnosis_cols:
        if col in row.index and not pd.isna(row[col]):
            if row[col] in [1, 2, 3, 4, 5, 6, 7]:
                has_diagnosis = True
                diagnoses.append(f"{col}={row[col]}")
    
    # Check autism diagnosis columns
    for col in autism_diagnosis_cols:
        if col in row.index and not pd.isna(row[col]):
            has_diagnosis = True
            diagnoses.append(f"{col}=autism")
    
    expected = row['any_condition_target'] == 1
    status = "✓" if has_diagnosis == expected else "✗"
    print(f"Person {idx}: {status} (has_diagnosis={has_diagnosis}, expected={expected}, diagnoses={diagnoses})")

# Check the first few people from each group specifically
print(f"\n=== GROUP VERIFICATION ===")
healthy_sample = balanced_data[balanced_data['any_condition_target'] == 0].head(3)
diagnosis_sample = balanced_data[balanced_data['any_condition_target'] == 1].head(3)

print("Healthy controls (should have no diagnoses):")
for idx, row in healthy_sample.iterrows():
    diagnoses = []
    for col in diagnosis_cols[:3]:
        if not pd.isna(row[col]) and row[col] not in [8, 9]:
            diagnoses.append(f"{col}={row[col]}")
    print(f"  Person {idx}: {diagnoses}")

print("\nDiagnosis cases (should have at least one diagnosis):")
for idx, row in diagnosis_sample.iterrows():
    diagnoses = []
    for col in diagnosis_cols[:3]:
        if not pd.isna(row[col]) and row[col] not in [8, 9]:
            diagnoses.append(f"{col}={row[col]}")
    print(f"  Person {idx}: {diagnoses}")

# 4.5 diagnosis breakdown analysis 

In [ ]:
# Comprehensive breakdown and verification of the balanced dataset
print("=== COMPREHENSIVE DATASET BREAKDOWN ===")

# Load the balanced dataset
balanced_data = pd.read_csv('/Users/eb2007/playground/bullpy/c4_experiments/data/processed/data_c4_any_condition_balanced.csv')

print(f"Balanced dataset shape: {balanced_data.shape}")
print(f"Target distribution: {balanced_data['any_condition_target'].value_counts()}")

# Get diagnosis columns
diagnosis_cols = [col for col in balanced_data.columns if col.startswith('diagnosis_') and col != 'diagnosis_9']
autism_diagnosis_cols = [col for col in balanced_data.columns if col.startswith('autism_diagnosis_')]

print(f"\nDiagnosis columns: {diagnosis_cols}")
print(f"Autism diagnosis columns: {autism_diagnosis_cols}")

# Condition mapping
condition_map = {
    1: 'ADHD',
    2: 'Autism Spectrum Disorder', 
    3: 'Bipolar Disorder',
    4: 'Depression',
    5: 'Learning disability',
    6: 'Obsessive-Compulsive Disorder',
    7: 'Schizophrenia',
    8: 'Prefer not to say',
    9: 'No diagnosis'
}

# Function to identify people with each condition (ONLY in diagnosis group)
def identify_condition_cases(data, condition_value, condition_name):
    """Identify people with a specific condition ONLY in the diagnosis group"""
    condition_mask = pd.Series([False] * len(data))
    
    # Only check people in the diagnosis group
    diagnosis_group_mask = data['any_condition_target'] == 1
    
    # Check main diagnosis columns
    for col in diagnosis_cols:
        if col in data.columns:
            # Only count if person is in diagnosis group AND has the condition
            condition_mask = condition_mask | (diagnosis_group_mask & (data[col] == condition_value))
    
    # Check autism diagnosis columns (for autism only)
    if condition_value == 2:  # Autism
        for col in autism_diagnosis_cols:
            if col in data.columns:
                # Only count if person is in diagnosis group AND has autism diagnosis
                condition_mask = condition_mask | (diagnosis_group_mask & data[col].notna())
    
    return condition_mask

# Count each condition
print(f"\n=== CONDITION BREAKDOWN IN DIAGNOSIS GROUP ===")
condition_counts = {}
for value, name in condition_map.items():
    if value not in [8, 9]:  # Skip "prefer not to say" and "no diagnosis"
        condition_mask = identify_condition_cases(balanced_data, value, name)
        condition_counts[name] = condition_mask.sum()
        print(f"{name}: {condition_mask.sum()} cases")

# Check for people with multiple conditions
print(f"\n=== MULTIPLE CONDITION ANALYSIS ===")

# Create condition flags for each person (ONLY in diagnosis group)
condition_flags = {}
for value, name in condition_map.items():
    if value not in [8, 9]:
        condition_flags[name] = identify_condition_cases(balanced_data, value, name)

# Count how many conditions each person has
balanced_data['condition_count'] = sum(condition_flags.values())

# Analyze the diagnosis group
diagnosis_group = balanced_data[balanced_data['any_condition_target'] == 1]
print(f"Diagnosis group condition count distribution:")
print(diagnosis_group['condition_count'].value_counts().sort_index())

# Check for people with multiple conditions
multiple_conditions = diagnosis_group[diagnosis_group['condition_count'] > 1]
print(f"\nPeople with multiple conditions: {len(multiple_conditions)} ({len(multiple_conditions)/len(diagnosis_group)*100:.1f}%)")

# Verify healthy controls have NO conditions
print(f"\n=== HEALTHY CONTROLS VERIFICATION ===")
healthy_group = balanced_data[balanced_data['any_condition_target'] == 0]
healthy_with_conditions = healthy_group[healthy_group['condition_count'] > 0]
print(f"Healthy controls with any condition: {len(healthy_with_conditions)}")

if len(healthy_with_conditions) > 0:
    print("WARNING: Healthy controls have conditions!")
    print("Sample healthy controls with conditions:")
    for idx, row in healthy_with_conditions.head(3).iterrows():
        conditions = []
        for value, name in condition_map.items():
            if value not in [8, 9]:
                if condition_flags[name][idx]:
                    conditions.append(name)
        print(f"  Person {idx}: {conditions}")
else:
    print("SUCCESS: All healthy controls have no conditions")

# Check autism specifically
autism_mask = identify_condition_cases(balanced_data, 2, 'Autism Spectrum Disorder')
print(f"\n=== AUTISM SPECIFIC ANALYSIS ===")
print(f"Total autism cases: {autism_mask.sum()}")
print(f"Autism in diagnosis group: {autism_mask[balanced_data['any_condition_target'] == 1].sum()}")
print(f"Autism in healthy group: {autism_mask[balanced_data['any_condition_target'] == 0].sum()}")

# Verify the logic - check some examples
print(f"\n=== VERIFICATION EXAMPLES ===")
print("Sample diagnosis cases with their conditions:")
sample_diagnosis = diagnosis_group.head(5)
for idx, row in sample_diagnosis.iterrows():
    conditions = []
    for value, name in condition_map.items():
        if value not in [8, 9]:
            if condition_flags[name][idx]:
                conditions.append(name)
    print(f"Person {idx}: {conditions} (count: {row['condition_count']})")

# Check autism diagnosis columns specifically
print(f"\n=== AUTISM DIAGNOSIS COLUMN ANALYSIS ===")
for col in autism_diagnosis_cols:
    if col in balanced_data.columns:
        print(f"\n{col} value counts:")
        value_counts = balanced_data[col].value_counts()
        print(value_counts)
        
        # Map autism diagnosis values
        autism_diagnosis_map = {
            1: 'Autism (classical autism)',
            2: 'Asperger Syndrome (AS)',
            3: 'Other'
        }
        
        print(f"{col} mapped to autism types:")
        for value, count in value_counts.items():
            if not pd.isna(value):
                autism_type = autism_diagnosis_map.get(int(value), f'Unknown ({value})')
                print(f"  {autism_type}: {count}")

# Final summary
print(f"\n=== FINAL SUMMARY ===")
print(f"Dataset: {balanced_data.shape[0]} participants")
print(f"Healthy controls: {len(healthy_group)} ({len(healthy_group)/len(balanced_data)*100:.1f}%)")
print(f"Diagnosis cases: {len(diagnosis_group)} ({len(diagnosis_group)/len(balanced_data)*100:.1f}%)")
print(f"Multiple conditions: {len(multiple_conditions)} ({len(multiple_conditions)/len(diagnosis_group)*100:.1f}% of diagnosis group)")

# Check for any remaining issues
print(f"\n=== QUALITY CHECKS ===")
print(f"Missing values in target: {balanced_data['any_condition_target'].isnull().sum()}")
print(f"Target values: {balanced_data['any_condition_target'].unique()}")

# Verify no cross-contamination
healthy_with_diagnosis = healthy_group[healthy_group['condition_count'] > 0]
diagnosis_without_condition = diagnosis_group[diagnosis_group['condition_count'] == 0]

print(f"Healthy controls with conditions: {len(healthy_with_diagnosis)}")
print(f"Diagnosis cases without conditions: {len(diagnosis_without_condition)}")

if len(healthy_with_diagnosis) == 0 and len(diagnosis_without_condition) == 0:
    print("PERFECT: No cross-contamination between groups")
else:
    print("ISSUE: Cross-contamination detected")

# FE - basic

In [ ]:
# Create a copy for feature engineering
df = balanced_data.copy()

print("Creating basic features...")

def create_aggregate_features(df, prefix, n_items):
    """Create aggregate features for questionnaire items"""
    item_cols = [f"{prefix}_{i}" for i in range(1, n_items+1) if f"{prefix}_{i}" in df.columns]
    if item_cols:
        df[f"{prefix}_total"] = df[item_cols].sum(axis=1)
    return df

# Create questionnaire totals
for prefix, n_items in [('eq', 10), ('aq', 10), ('sqr', 10), ('spq', 10)]:
    df = create_aggregate_features(df, prefix, n_items)

# D-score (Empathy - Social Responsiveness)
if 'eq_total' in df.columns and 'sqr_total' in df.columns:
    df['d_score'] = df['eq_total'] - df['sqr_total']

# Age interactions
if 'age' in df.columns and 'aq_total' in df.columns:
    df['age_x_aq'] = df['age'] * df['aq_total']
if 'age' in df.columns and 'eq_total' in df.columns:
    df['age_x_eq'] = df['age'] * df['eq_total']

# Trait interactions
if 'aq_total' in df.columns and 'eq_total' in df.columns:
    df['aq_eq_interaction'] = df['aq_total'] * df['eq_total']

# Ratios for cognitive profiles
if 'eq_total' in df.columns and 'sqr_total' in df.columns:
    df['eq_sqr_ratio'] = df['eq_total'] / (df['sqr_total'] + 1e-8)
if 'aq_total' in df.columns and 'eq_total' in df.columns:
    df['aq_eq_ratio'] = df['aq_total'] / (df['eq_total'] + 1e-8)

# Log transformations
if 'aq_total' in df.columns:
    df['log_aq_total'] = np.log1p(np.clip(df['aq_total'], a_min=0, a_max=None))
if 'age' in df.columns:
    df['sqrt_age'] = np.sqrt(np.clip(df['age'], a_min=0, a_max=None))

# High trait flags
if 'aq_total' in df.columns:
    df['high_aq'] = (df['aq_total'] > 32).astype(int)
if 'eq_total' in df.columns:
    df['low_eq'] = (df['eq_total'] < 30).astype(int)

print(f"Features after basic engineering: {len(df.columns)}")

# 6. FE - grounded in literature 

In [ ]:
print("Creating additional features based on scientific literature...")

# Age-related features (developmental considerations)
if 'age' in df.columns:
    df['age_squared'] = df['age'] ** 2
    df['age_cubed'] = df['age'] ** 3
    df['log_age'] = np.log1p(df['age'])
    
    # Age groups for developmental stages
    df['age_group'] = pd.cut(df['age'], bins=[0, 18, 25, 35, 50, 100], 
                             labels=['adolescent', 'young_adult', 'adult', 'middle_age', 'senior'])

# Sex-related features (mental health prevalence differs by sex)
if 'sex' in df.columns:
    # Sex-specific patterns
    df['sex_x_aq'] = df['sex'] * df['aq_total']
    df['sex_x_eq'] = df['sex'] * df['eq_total']
    df['sex_x_sqr'] = df['sex'] * df['sqr_total']

# Questionnaire subdomain features (based on factor analysis literature)
if all(f'aq_{i}' in df.columns for i in range(1, 11)):
    # AQ subdomains (Baron-Cohen et al., 2001)
    df['aq_social_skills'] = df[['aq_1', 'aq_7', 'aq_8', 'aq_9', 'aq_10']].sum(axis=1)
    df['aq_attention_switching'] = df[['aq_2', 'aq_4', 'aq_6']].sum(axis=1)
    df['aq_attention_detail'] = df[['aq_3', 'aq_5']].sum(axis=1)

if all(f'eq_{i}' in df.columns for i in range(1, 11)):
    # EQ subdomains (Baron-Cohen & Wheelwright, 2004)
    df['eq_cognitive'] = df[['eq_1', 'eq_3', 'eq_5', 'eq_7', 'eq_9']].sum(axis=1)
    df['eq_affective'] = df[['eq_2', 'eq_4', 'eq_6', 'eq_8', 'eq_10']].sum(axis=1)

if all(f'sqr_{i}' in df.columns for i in range(1, 11)):
    # SQR subdomains (Constantino & Gruber, 2005)
    df['sqr_social_awareness'] = df[['sqr_1', 'sqr_2', 'sqr_3']].sum(axis=1)
    df['sqr_social_cognition'] = df[['sqr_4', 'sqr_5', 'sqr_6']].sum(axis=1)
    df['sqr_social_communication'] = df[['sqr_7', 'sqr_8', 'sqr_9']].sum(axis=1)
    df['sqr_social_motivation'] = df[['sqr_10']].sum(axis=1)

# SPQ subdomain features (Raine, 1991)
if all(f'spq_{i}' in df.columns for i in range(1, 11)):
    df['spq_cognitive_perceptual'] = df[['spq_1', 'spq_2', 'spq_3', 'spq_4']].sum(axis=1)
    df['spq_interpersonal'] = df[['spq_5', 'spq_6', 'spq_7', 'spq_8']].sum(axis=1)
    df['spq_disorganized'] = df[['spq_9', 'spq_10']].sum(axis=1)

print(f"Features after scientific features: {len(df.columns)}")

# FE - grounded in stats

In [ ]:
print("Creating statistical features...")

# Z-scores for questionnaire totals
for col in ['aq_total', 'eq_total', 'sqr_total', 'spq_total']:
    if col in df.columns:
        df[f'{col}_zscore'] = (df[col] - df[col].mean()) / df[col].std()

# Percentile ranks
for col in ['aq_total', 'eq_total', 'sqr_total', 'spq_total']:
    if col in df.columns:
        df[f'{col}_percentile'] = df[col].rank(pct=True)

# Extreme value indicators
for col in ['aq_total', 'eq_total', 'sqr_total', 'spq_total']:
    if col in df.columns:
        df[f'{col}_extreme_high'] = (df[col] > df[col].quantile(0.95)).astype(int)
        df[f'{col}_extreme_low'] = (df[col] < df[col].quantile(0.05)).astype(int)

# Three-way interactions
if all(col in df.columns for col in ['aq_total', 'eq_total', 'sqr_total']):
    df['aq_eq_sqr_interaction'] = df['aq_total'] * df['eq_total'] * df['sqr_total']

# Quadratic terms
for col in ['aq_total', 'eq_total', 'sqr_total']:
    if col in df.columns:
        df[f'{col}_squared'] = df[col] ** 2

# Cross-ratio features
if all(col in df.columns for col in ['aq_total', 'eq_total', 'sqr_total']):
    df['aq_eq_cross_ratio'] = df['aq_total'] / (df['eq_total'] + 1e-8)
    df['aq_sqr_cross_ratio'] = df['aq_total'] / (df['sqr_total'] + 1e-8)
    df['eq_sqr_cross_ratio'] = df['eq_total'] / (df['sqr_total'] + 1e-8)

print(f"Features after statistical features: {len(df.columns)}")

# FE - grounded in clincal evidence

In [ ]:
print("Creating clinical features...")

# Mental health screening thresholds (based on literature)
if 'aq_total' in df.columns:
    df['aq_above_threshold'] = (df['aq_total'] > 26).astype(int)  # Baron-Cohen et al., 2001
    df['aq_high_threshold'] = (df['aq_total'] > 32).astype(int)   # Clinical threshold

# Empathy deficits
if 'eq_total' in df.columns:
    df['eq_below_threshold'] = (df['eq_total'] < 30).astype(int)  # Baron-Cohen & Wheelwright, 2004

# Social responsiveness deficits
if 'sqr_total' in df.columns:
    df['sqr_above_threshold'] = (df['sqr_total'] > 60).astype(int)  # Constantino & Gruber, 2005

# Schizotypy features
if 'spq_total' in df.columns:
    df['spq_above_threshold'] = (df['spq_total'] > 22).astype(int)  # Raine, 1991

# ============================================================================
# DEMOGRAPHIC AND OCCUPATION FEATURES
# ============================================================================

# Education level features
if 'education' in df.columns:
    df['high_education'] = (df['education'] >= 3).astype(int)  # Undergraduate or higher
    df['low_education'] = (df['education'] <= 2).astype(int)   # High school or lower

# Occupation features (if available)
if 'occupation' in df.columns:
    # STEM occupations (based on occupation codes)
    stem_occupations = [3, 5, 21]  # Computers & IT, Engineering, Scientific & Technical
    df['is_stem_occupation'] = df['occupation'].isin(stem_occupations).astype(int)
    
    if 'is_stem_occupation' in df.columns:
        df['stem_x_aq'] = df['is_stem_occupation'] * df['aq_total']
        df['stem_x_eq'] = df['is_stem_occupation'] * df['eq_total']

# Sex category features (one-hot encoded)
if 'sex' in df.columns:
    # Create binary features for each sex category
    for sex_value in [1, 2, 3, 4]:  # Male, Female, Other, Prefer not to say
        df[f'sex_{sex_value}'] = (df['sex'] == sex_value).astype(int)
        if f'sex_{sex_value}' in df.columns:
            df[f'sex_{sex_value}_x_aq'] = df[f'sex_{sex_value}'] * df['aq_total']

# Handedness features
if 'handedness' in df.columns:
    df['left_handed'] = (df['handedness'] == 2).astype(int)
    df['ambidextrous'] = (df['handedness'] == 3).astype(int)

print(f"Features after clinical features: {len(df.columns)}")

# 9. feature selection and data prep

In [ ]:
print("Feature selection and preparation...")

# Remove target variable FIRST
X = df.drop(columns=['any_condition_target'])
y = df['any_condition_target']

# CRITICAL: Remove ALL diagnosis-related features that could leak
diagnosis_leakage_features = [col for col in X.columns if any(term in col.lower() for term in 
                          ['diagnosis', 'autism_diagnosis', 'adhd', 'autism', 'bipolar', 'depression', 
                           'learning', 'ocd', 'schizophrenia', 'risk_score', 'target', 'condition_count'])]
if diagnosis_leakage_features:
    print(f"Removing diagnosis leakage features: {diagnosis_leakage_features}")
    X = X.drop(columns=diagnosis_leakage_features)

# Remove non-numeric columns
X = X.select_dtypes(include=[np.number])

# Remove constant features
constant_features = [col for col in X.columns if X[col].nunique() == 1]
X = X.drop(columns=constant_features)

# Final leakage check
print(f"Final feature set: {X.shape[1]} features")
print(f"Target variable: {y.name}")
print(f"Sample features: {list(X.columns[:10])}")

# CRITICAL: Verify no leakage
print(f"Target in features: {'any_condition_target' in X.columns}")
print(f"Any 'diagnosis' in features: {any('diagnosis' in col.lower() for col in X.columns)}")
print(f"Any 'risk' in features: {any('risk' in col.lower() for col in X.columns)}")
print(f"Any 'target' in features: {any('target' in col.lower() for col in X.columns)}")
print(f"condition_count in features: {'condition_count' in X.columns}")

# Handle missing values
X = X.fillna(X.mean())

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Target distribution - Train: {y_train.mean():.3f}, Test: {y_test.mean():.3f}")

# model definitions 

In [ ]:
print("Defining optimized models for any condition prediction...")

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=50, max_depth=6, random_state=42),
    'Extra Trees': ExtraTreesClassifier(n_estimators=50, max_depth=10, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Naive Bayes': GaussianNB(),
    'Decision Tree': DecisionTreeClassifier(max_depth=8, random_state=42),
    'XGBoost': xgb.XGBClassifier(n_estimators=50, max_depth=6, random_state=42, n_jobs=-1),
    'LightGBM': lgb.LGBMClassifier(n_estimators=50, max_depth=6, random_state=42, n_jobs=-1)
}

# model training and eval 

In [ ]:
print("Training and evaluating optimized models...")

results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    try:
        # Train model
        model.fit(X_train_scaled, y_train)
        
        # Predictions
        y_pred = model.predict(X_test_scaled)
        y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
        
        # Metrics
        accuracy = accuracy_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_pred_proba)
        f1 = f1_score(y_test, y_pred)
        
        # Cross-validation (reduced to 3-fold for speed)
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=3, scoring='accuracy', n_jobs=-1)
        cv_f1_scores = cross_val_score(model, X_train_scaled, y_train, cv=3, scoring='f1', n_jobs=-1)
        
        results[name] = {
            'accuracy': accuracy,
            'auc': auc,
            'f1': f1,
            'cv_mean': cv_scores.mean(),
            'cv_std': cv_scores.std(),
            'cv_f1_mean': cv_f1_scores.mean(),
            'cv_f1_std': cv_f1_scores.std(),
            'model': model
        }
        
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  AUC: {auc:.4f}")
        print(f"  F1 Score: {f1:.4f}")
        print(f"  CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
        print(f"  CV F1: {cv_f1_scores.mean():.4f} (+/- {cv_f1_scores.std() * 2:.4f})")
        
    except Exception as e:
        print(f"  Error: {e}")
        results[name] = {'error': str(e)}

# results summary

In [ ]:
print("\n" + "="*60)
print("MODEL PERFORMANCE SUMMARY")
print("="*60)

# Create results dataframe
results_df = pd.DataFrame([
    {
        'Model': name,
        'Accuracy': result.get('accuracy', np.nan),
        'AUC': result.get('auc', np.nan),
        'F1': result.get('f1', np.nan),
        'CV_Accuracy': result.get('cv_mean', np.nan),
        'CV_Std': result.get('cv_std', np.nan)
    }
    for name, result in results.items() if 'error' not in result
]).sort_values('AUC', ascending=False)

print(results_df.to_string(index=False))

# Find best model
best_model_name = results_df.iloc[0]['Model']
best_model = results[best_model_name]['model']

print(f"\nBEST MODEL: {best_model_name}")
print(f"Best AUC: {results_df.iloc[0]['AUC']:.4f}")
print(f"Best Accuracy: {results_df.iloc[0]['Accuracy']:.4f}")
print(f"Best F1: {results_df.iloc[0]['F1']:.4f}")